In [ ]:
!pip install scikit-learn pandas statsmodels pyarrow -q


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.stats import f as f_dist
import warnings
warnings.filterwarnings("ignore")

PARQUET_PATH = os.environ.get("EEG_PARQUET", "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw/eeg_preprocessed_4ch_raw.parquet")
CSV_OUT = "granger_classical_offset_results.csv"

LIMIT_PATIENTS = None
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
OFFSET = 1500

LAGS_TO_TEST = [5, 10, 20, 30, 40, 50]
MAX_LAG = max(LAGS_TO_TEST)
TEST_PAIRS = [('P3', 'Fp1'), ('Fp1', 'P3'), ('P4', 'Fp2'), ('Fp2', 'P4')]

def window_starts_with_history(total_len: int, eval_len: int, history_len: int, n_win: int) -> np.ndarray:
    start_idx = max(history_len, OFFSET)
    if total_len < eval_len + start_idx:
        raise ValueError("Signal too short.")
    hi = total_len - eval_len
    return np.linspace(start_idx, hi, n_win, dtype=int)

def manual_granger_f_stat(y_chunk, x_chunk, lag):
    n = len(y_chunk)
    y_target = y_chunk[lag:]
    X_restricted = np.ones((n - lag, 1))
    for l in range(1, lag + 1):
        X_restricted = np.column_stack([X_restricted, y_chunk[lag-l:-l]])
    X_full = X_restricted.copy()
    for l in range(1, lag + 1):
        X_full = np.column_stack([X_full, x_chunk[lag-l:-l]])
    reg_restricted = LinearRegression().fit(X_restricted, y_target)
    reg_full = LinearRegression().fit(X_full, y_target)
    pred_restricted = reg_restricted.predict(X_restricted)
    pred_full = reg_full.predict(X_full)
    rss_restricted = np.sum((y_target - pred_restricted) ** 2)
    rss_full = np.sum((y_target - pred_full) ** 2)
    k = lag
    obs = len(y_target)
    if rss_full > 0:
        f_stat = ((rss_restricted - rss_full) / k) / (rss_full / (obs - 2*k - 1))
        p_val = f_dist.sf(f_stat, k, obs - 2*k - 1)
    else:
        f_stat, p_val = 0.0, 1.0
    return f_stat, p_val

def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)): return np.asarray(cell, dtype=np.float64)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float64)

print("Starting Classical Granger Causality (with 3s offset)...")
df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
results = []
done = 0
eval_window_size = CONTEXT_LENGTH + HORIZON_LENGTH

for _, row in df_eval.iterrows():
    sid = row["subject_id"]
    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    for x_col, y_col in TEST_PAIRS:
        x_signal = series_to_numpy(row[x_col])
        y_signal = series_to_numpy(row[y_col])
        starts = window_starts_with_history(len(y_signal), eval_window_size, MAX_LAG, N_WINDOWS)
        for win_idx, start_idx in enumerate(starts):
            chunk_start = start_idx - MAX_LAG
            chunk_end = start_idx + eval_window_size
            x_full_chunk = x_signal[chunk_start:chunk_end]
            y_full_chunk = y_signal[chunk_start:chunk_end]
            for current_lag in LAGS_TO_TEST:
                offset = MAX_LAG - current_lag
                x_chunk = x_full_chunk[offset:]
                y_chunk = y_full_chunk[offset:]
                f_manual, p_manual = manual_granger_f_stat(y_chunk, x_chunk, current_lag)
                results.append({
                    "subject_id": sid, "group": grp, "covariate_X": x_col, "target_Y": y_col,
                    "window_index": win_idx, "lag_steps": current_lag, "lag_ms": int(current_lag * 2),
                    "f_stat": f_manual, "p_value": p_manual
                })
    done += 1
    if done % 10 == 0 or done == len(df_eval): print(f"Processed: {done} / {len(df_eval)}")

df_results = pd.DataFrame(results)
df_results.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")
